In [ ]:
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# List all files in your Drive root
!ls "/content/drive/My Drive/"

# List only zip files in the Drive
!ls "/content/drive/My Drive/" | grep ".zip"


'Colab Notebooks'		   'Vitamin Deficiency Detection System'
 fouji.zip			    vitamin_models
'vitamin_deficiency_data (2).zip'   VQA_Project.zip
 vitamin_deficiency_dataa.zip
fouji.zip
vitamin_deficiency_data (2).zip
vitamin_deficiency_dataa.zip
VQA_Project.zip


In [ ]:
!ls "/content/drive/My Drive/" | grep "vitamin_deficiency_data (2).zip"


vitamin_deficiency_data (2).zip


In [ ]:
# Unzip only the second zip file
!unzip -q "/content/drive/My Drive/vitamin_deficiency_data (2).zip" -d "/content/vitamin_deficiency_data_2"

# Check the contents
!ls "/content/vitamin_deficiency_data_2"


angular_cheilitis  healthy_elbows  healthy_mouth   ulcer
bitot_spots	   healthy_eye	   healthy_tongue  vitamin_deficiency_data
glossitis	   healthy_lips    phrynoderma


In [ ]:
base_path = "/content/vitamin_deficiency_data_2/vitamin_deficiency_data"


In [ ]:
!mv /content/vitamin_deficiency_data_2/vitamin_deficiency_data/* /content/vitamin_deficiency_data_2/
!rm -r /content/vitamin_deficiency_data_2/vitamin_deficiency_data


mv: cannot stat '/content/vitamin_deficiency_data_2/vitamin_deficiency_data/*': No such file or directory
rm: cannot remove '/content/vitamin_deficiency_data_2/vitamin_deficiency_data': No such file or directory


In [ ]:
# Remove hyperpigmented folder
!rm -r "/content/vitamin_deficiency_data_2/hyperpigmented"

# Remove healthy_knee folder
!rm -r "/content/vitamin_deficiency_data_2/healthy_knee"


In [ ]:
!ls -d /content/vitamin_deficiency_data_2/*/


/content/vitamin_deficiency_data_2/angular_cheilitis/
/content/vitamin_deficiency_data_2/bitot_spots/
/content/vitamin_deficiency_data_2/glossitis/
/content/vitamin_deficiency_data_2/healthy_elbows/
/content/vitamin_deficiency_data_2/healthy_eye/
/content/vitamin_deficiency_data_2/healthy_lips/
/content/vitamin_deficiency_data_2/healthy_mouth/
/content/vitamin_deficiency_data_2/healthy_tongue/
/content/vitamin_deficiency_data_2/phrynoderma/
/content/vitamin_deficiency_data_2/ulcer/


In [ ]:
import os
import shutil

# Original data folder
base_dir = "/content/vitamin_deficiency_data_2"

# New dataset folder for symptom-based classification
dataset_dir = "/content/vitamin_deficiency_symptoms"
os.makedirs(dataset_dir, exist_ok=True)

# List of all classes (symptoms + healthy)
classes = [
    "angular_cheilitis",
    "glossitis",
    "ulcer",
    "bitot_spots",
    "phrynoderma",
    "healthy_elbows",
    "healthy_eye",
    "healthy_lips",
    "healthy_mouth",
    "healthy_tongue"
]

# Create train/test folders for each class
for split in ["train", "test"]:
    for cls in classes:
        os.makedirs(os.path.join(dataset_dir, split, cls), exist_ok=True)

# Copy images from original folders to the new structure
for cls in classes:
    for split in ["train", "test"]:
        src_path = os.path.join(base_dir, cls, split)
        dst_path = os.path.join(dataset_dir, split, cls)
        if os.path.exists(src_path):
            for file in os.listdir(src_path):
                shutil.copy(os.path.join(src_path, file), dst_path)

print("Dataset reorganized into symptom-based classes successfully!")


Dataset reorganized into symptom-based classes successfully!


In [ ]:
import os

dataset_dir = "/content/vitamin_deficiency_symptoms"

for split in ["train", "test"]:
    print(f"\nContents of {split} folder:")
    split_path = os.path.join(dataset_dir, split)
    classes = [d for d in os.listdir(split_path) if os.path.isdir(os.path.join(split_path, d))]
    print(classes)



Contents of train folder:
['angular_cheilitis', 'glossitis', 'healthy_mouth', 'healthy_tongue', 'phrynoderma', 'bitot_spots', 'healthy_lips', 'ulcer', 'healthy_elbows', 'healthy_eye']

Contents of test folder:
['angular_cheilitis', 'glossitis', 'healthy_mouth', 'healthy_tongue', 'phrynoderma', 'bitot_spots', 'healthy_lips', 'ulcer', 'healthy_elbows', 'healthy_eye']


In [ ]:
import os

dataset_dir = "/content/vitamin_deficiency_symptoms"

for split in ["train", "test"]:
    total_images = 0
    split_path = os.path.join(dataset_dir, split)
    for cls in os.listdir(split_path):
        cls_path = os.path.join(split_path, cls)
        if os.path.isdir(cls_path):
            total_images += len(os.listdir(cls_path))
    print(f"Total images in {split}: {total_images}")


Total images in train: 2339
Total images in test: 226


In [ ]:
import os

dataset_dir = "/content/vitamin_deficiency_symptoms"

for split in ["train", "test"]:
    print(f"\nNumber of images in each class in {split}:")
    split_path = os.path.join(dataset_dir, split)
    for cls in sorted(os.listdir(split_path)):
        cls_path = os.path.join(split_path, cls)
        if os.path.isdir(cls_path):
            num_images = len(os.listdir(cls_path))
            print(f"{cls}: {num_images}")



Number of images in each class in train:
angular_cheilitis: 270
bitot_spots: 270
glossitis: 270
healthy_elbows: 180
healthy_eye: 270
healthy_lips: 179
healthy_mouth: 180
healthy_tongue: 180
phrynoderma: 270
ulcer: 270

Number of images in each class in test:
angular_cheilitis: 24
bitot_spots: 10
glossitis: 10
healthy_elbows: 11
healthy_eye: 37
healthy_lips: 28
healthy_mouth: 19
healthy_tongue: 16
phrynoderma: 32
ulcer: 39


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt


In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((128,128)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
])

test_transform = transforms.Compose([
    transforms.Resize((128,128)),
    transforms.ToTensor(),
])


In [ ]:
import os
import shutil

base_dir = "/content/vitamin_deficiency_data_2"

# Create global train and test folders
global_train = "/content/train"
global_test = "/content/test"

os.makedirs(global_train, exist_ok=True)
os.makedirs(global_test, exist_ok=True)

classes = os.listdir(base_dir)

for cls in classes:
    class_path = os.path.join(base_dir, cls)
    if not os.path.isdir(class_path):
        continue

    # Create class folders in global train/test
    os.makedirs(os.path.join(global_train, cls), exist_ok=True)
    os.makedirs(os.path.join(global_test, cls), exist_ok=True)

    # Move train images
    train_src = os.path.join(class_path, "train")
    if os.path.exists(train_src):
        for img in os.listdir(train_src):
            shutil.move(os.path.join(train_src, img), os.path.join(global_train, cls))

    # Move test images
    test_src = os.path.join(class_path, "test")
    if os.path.exists(test_src):
        for img in os.listdir(test_src):
            shutil.move(os.path.join(test_src, img), os.path.join(global_test, cls))

print("Dataset reorganized successfully.")


Dataset reorganized successfully.


In [ ]:
train_dir = "/content/train"
test_dir = "/content/test"

train_data = datasets.ImageFolder(train_dir, transform=train_transform)
test_data = datasets.ImageFolder(test_dir, transform=test_transform)


In [ ]:
!ls /content/train
!ls /content/test


angular_cheilitis  glossitis	   healthy_eye	 healthy_mouth	 phrynoderma
bitot_spots	   healthy_elbows  healthy_lips  healthy_tongue  ulcer
angular_cheilitis  glossitis	   healthy_eye	 healthy_mouth	 phrynoderma
bitot_spots	   healthy_elbows  healthy_lips  healthy_tongue  ulcer


In [ ]:

# ----- UPDATED PATHS -----
train_dir = "/content/train"
test_dir = "/content/test"

# ----- LOAD DATA -----
train_data = datasets.ImageFolder(train_dir, transform=train_transform)
test_data = datasets.ImageFolder(test_dir, transform=test_transform)

train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
test_loader = DataLoader(test_data, batch_size=32, shuffle=False)

print("Classes found:", train_data.classes)
print("Number of classes:", len(train_data.classes))

print("Train samples:", len(train_data))
print("Test samples:", len(test_data))

Classes found: ['angular_cheilitis', 'bitot_spots', 'glossitis', 'healthy_elbows', 'healthy_eye', 'healthy_lips', 'healthy_mouth', 'healthy_tongue', 'phrynoderma', 'ulcer']
Number of classes: 10
Train samples: 2596
Test samples: 226


In [ ]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Transforms
train_transform = transforms.Compose([
    transforms.Resize((128,128)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
])

test_transform = transforms.Compose([
    transforms.Resize((128,128)),
    transforms.ToTensor(),
])

train_dir = "/content/train"
test_dir  = "/content/test"

train_data = datasets.ImageFolder(train_dir, transform=train_transform)
test_data  = datasets.ImageFolder(test_dir, transform=test_transform)

train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
test_loader  = DataLoader(test_data, batch_size=32, shuffle=False)

print("Classes:", train_data.classes)
print("Train samples:", len(train_data))
print("Test samples:", len(test_data))


Classes: ['angular_cheilitis', 'bitot_spots', 'glossitis', 'healthy_elbows', 'healthy_eye', 'healthy_lips', 'healthy_mouth', 'healthy_tongue', 'phrynoderma', 'ulcer']
Train samples: 2596
Test samples: 226


In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class BaselineCNN(nn.Module):
    def __init__(self, num_classes):
        super(BaselineCNN, self).__init__()
        self.conv1 = nn.Conv2d(3, 16, 3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, 3, padding=1)
        self.conv3 = nn.Conv2d(32, 64, 3, padding=1)
        self.pool = nn.MaxPool2d(2,2)
        self.fc1 = nn.Linear(64*16*16, 256)
        self.fc2 = nn.Linear(256, num_classes)
        self.dropout = nn.Dropout(0.3)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = self.pool(F.relu(self.conv3(x)))
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x

num_classes = len(train_data.classes)
cnn_model = BaselineCNN(num_classes).to(device)


In [ ]:
import torch.optim as optim

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(cnn_model.parameters(), lr=0.0005)
epochs = 10

from tqdm import tqdm

for epoch in range(epochs):
    cnn_model.train()
    total_loss, correct, total = 0, 0, 0
    for images, labels in tqdm(train_loader, leave=False):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = cnn_model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        _, pred = torch.max(outputs, 1)
        correct += (pred == labels).sum().item()
        total += labels.size(0)

    train_acc = 100*correct/total
    avg_loss = total_loss/len(train_loader)
    print(f"Epoch {epoch+1}/{epochs} | Loss: {avg_loss:.4f} | Train Accuracy: {train_acc:.2f}%")



Epoch 1/10 | Loss: 1.7054 | Train Accuracy: 40.52%


Epoch 2/10 | Loss: 0.9971 | Train Accuracy: 66.87%


Epoch 3/10 | Loss: 0.7107 | Train Accuracy: 75.96%


Epoch 4/10 | Loss: 0.5795 | Train Accuracy: 80.51%


Epoch 5/10 | Loss: 0.5212 | Train Accuracy: 82.05%


Epoch 6/10 | Loss: 0.4307 | Train Accuracy: 85.02%


Epoch 7/10 | Loss: 0.3881 | Train Accuracy: 86.06%


Epoch 8/10 | Loss: 0.3348 | Train Accuracy: 88.25%


Epoch 9/10 | Loss: 0.3049 | Train Accuracy: 89.87%


Epoch 10/10 | Loss: 0.2539 | Train Accuracy: 90.83%


In [ ]:
cnn_model.eval()
correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = cnn_model(images)
        _, pred = torch.max(outputs,1)
        correct += (pred == labels).sum().item()
        total += labels.size(0)

test_acc = 100 * correct / total
print(f"📌 Test Accuracy: {test_acc:.2f}%")


📌 Test Accuracy: 74.34%
